# Notebook 02 — Preprocessing Pipeline

**Project:** Machine Learning-Based Intrusion Detection for Cloud Network Security  
**Author:** Dingaan Mahlatse Machethe | EC-Council University | ECCU500

---

Executes and validates the full preprocessing pipeline from `src/preprocess.py` (paper §5.2):

| Step | Operation | Paper Reference |
|------|-----------|----------------|
| 1 | Download NSL-KDD | §5.2 |
| 2 | Assign 42 column names | §5.2 |
| 3 | One-hot encode `protocol_type`, `service`, `flag` | §5.2 |
| 4 | MinMaxScaler normalisation | §5.2 |
| 5 | SMOTE oversampling | §5.2 |
| 6 | RFECV → top 25 features | §5.2 |
| 7 | Stratified 80/20 train-test split | §5.2 |
| 8 | Save `.npy` arrays to `data/` | §5.2 |
| 9 | Save `.pkl` artefacts to `results/models/` | §5.2 |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.preprocess import (
    download_nsl_kdd, encode_labels, one_hot_encode,
    normalise_features, apply_smote, select_features_rfecv,
    stratified_split, save_arrays, save_artifacts,
    N_FEATURES_TO_SELECT, TEST_SIZE,
)

sns.set_theme(style='whitegrid')
print('Pipeline modules loaded.')

## Step 1–2: Download & Assign Column Names

In [ ]:
df = download_nsl_kdd()
print(f'Raw shape: {df.shape}')
assert len(df.columns) == 42, f'Expected 42 columns, got {len(df.columns)}'
print('✓ 42 standard NSL-KDD column names assigned')
df.head(3)

## Step 2: Binary Label Encoding (Normal=0, Attack=1)

In [ ]:
df = encode_labels(df)
print(df['label'].value_counts())

fig, ax = plt.subplots(figsize=(6, 4))
df['label'].value_counts().plot(kind='bar', ax=ax, color=['#2ecc71','#e74c3c'])
ax.set_xticklabels(['Normal (0)','Attack (1)'], rotation=0)
ax.set_title('Binary Label Distribution — Before SMOTE\n(Paper §5.2: SMOTE applied to handle class imbalance)')
plt.tight_layout()
plt.show()

## Step 3: One-Hot Encoding (protocol_type, service, flag)

In [ ]:
n_before = len(df.columns)
df = one_hot_encode(df)
print(f'Columns before OHE: {n_before}')
print(f'Columns after OHE : {len(df.columns)}')
print(f'New OHE columns   : {len(df.columns) - n_before}')
print('✓ One-hot encoding complete')

## Step 4: MinMaxScaler Normalisation

In [ ]:
y = df['label'].values
X = df.drop(columns=['label']).values.astype('float32')
feature_names = list(df.drop(columns=['label']).columns)

X_scaled, scaler = normalise_features(X, fit=True)

print(f'Feature range before scaling: [{X.min():.1f}, {X.max():.1f}]')
print(f'Feature range after  scaling: [{X_scaled.min():.4f}, {X_scaled.max():.4f}]')
print('✓ MinMaxScaler normalisation complete')

## Step 5: SMOTE Oversampling

In [ ]:
X_resampled, y_resampled = apply_smote(X_scaled, y)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pre_counts = dict(zip(*np.unique(y, return_counts=True)))
post_counts = dict(zip(*np.unique(y_resampled, return_counts=True)))

for ax, counts, title in zip(axes, [pre_counts, post_counts], ['Before SMOTE','After SMOTE']):
    ax.bar(['Normal','Attack'], [counts.get(0,0), counts.get(1,0)], color=['#2ecc71','#e74c3c'])
    ax.set_title(title); ax.set_ylabel('Sample Count')
    for i, v in enumerate([counts.get(0,0), counts.get(1,0)]):
        ax.text(i, v + 100, f'{v:,}', ha='center')

plt.suptitle('SMOTE Class Balancing (Paper §5.2)', fontsize=11)
plt.tight_layout()
plt.savefig('../results/figures/02_smote_balancing.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6: RFECV Feature Selection (top 25 features)

In [ ]:
print(f'Target: {N_FEATURES_TO_SELECT} features (paper §5.2: "reduce to the 25 most informative features")')
print('Running RFECV... (this may take a few minutes)')

X_selected, selector = select_features_rfecv(X_resampled, y_resampled)

selected_features = [f for f, m in zip(feature_names, selector.support_) if m]
print(f'\nSelected {len(selected_features)} features:')
for i, f in enumerate(selected_features, 1):
    print(f'  {i:2d}. {f}')

In [ ]:
# Feature importance plot
importances = selector.estimator_.feature_importances_
selected_idx = [i for i, m in enumerate(selector.support_) if m]
top_imp = sorted(zip([feature_names[i] for i in selected_idx], importances), key=lambda x: -x[1])[:15]

fig, ax = plt.subplots(figsize=(10, 6))
names, vals = zip(*top_imp)
ax.barh(names[::-1], vals[::-1], color='#3498db')
ax.set_xlabel('Feature Importance Score')
ax.set_title('Top 15 Features by Importance (RFECV / Random Forest)\nRef: Paper Appendix B')
plt.tight_layout()
plt.savefig('../results/figures/02_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Stratified 80/20 Train-Test Split

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = stratified_split(X_selected, y_resampled)

# Re-fit scaler on training data only
X_train, scaler_final = normalise_features(X_train_raw, fit=True)
X_test, _ = normalise_features(X_test_raw, scaler=scaler_final, fit=False)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test : {X_test.shape} | y_test : {y_test.shape}')
print(f'Train split: {len(X_train)/len(X_selected)*100:.1f}%  (paper: 80%)')
print(f'Test  split: {len(X_test)/len(X_selected)*100:.1f}%  (paper: 20%)')

## Steps 8–9: Save Processed Data and Artefacts

In [ ]:
save_arrays(X_train, X_test, y_train, y_test)
save_artifacts(scaler_final, selector, feature_names)
print('✓ Processed arrays saved to data/')
print('✓ Scaler and RFECV selector saved to results/models/')

---
## Pipeline Summary

| Stage | Input Shape | Output Shape | Notes |
|-------|-------------|-------------|-------|
| Raw download | — | (148,517, 42) | 41 features + label |
| After OHE | (148,517, 42) | varies | +OHE columns for 3 cats |
| After SMOTE | varies | balanced | 50/50 class split |
| After RFECV | — | (·, 25) | Top 25 features (paper §5.2) |
| X_train | — | (·, 25) | 80% of SMOTE-balanced set |
| X_test | — | (·, 25) | 20% of SMOTE-balanced set |

**Next:** `03_model_training.ipynb` — train all 5 ML algorithms with paper-exact hyperparameters